# Phase 1b — LAUSD 2024-25: par ROI by kind of dollar (levels or growth)
**These results are associational, not causal.** See `src/returns_model.py` for the method.

Checks recorded on 2026-09-29:
- **CDE Current Expense of Education vs Phase 1.** Spending is $10,877M (CDE) vs $10,958M (ours), a +0.7% gap. Per ADA: $29,292 (CDE) vs $29,510 (ours). The ADA is identical (371,342.10). On 2026-10-02 we found that SACS K-12 ADA (336,447) leaves out the 34,895 ADA of charters reported in LAUSD's Fund 01, whose spending *is* in Fund 01. The pipeline now adds it back, so the gap is the 0.7% spending difference.
- **ESSA school-level total vs SACS General Fund.** $10,685M vs $10,958M, a 2.5% gap. ESSA's *federal* figure ($1,109M) includes the cafeteria fund (Fund 13, $492M federal), which the General Fund leaves out.
- **Robustness of the spending coefficients.** Checked with trimmed spending tails, weighting by students tested, and elementary schools only. State & local stays negative in every version, and federal always includes zero.

In [1]:
import sys; sys.path.insert(0, '../src')
import pandas as pd
import sacs_parser, sacs_statement, returns_model
from cds_lookup import CDS
cds = CDS.parse('19647330000000')
fmt = lambda v: f'${v/1e6:,.1f}M'

## 1. CDE's own per-ADA figure (validation of Phase 1)

In [2]:
cee = pd.read_excel('../data/raw/cde/currentexpense2425.xlsx', 'District', header=None, dtype=str)
cee[cee[2] == 'Los Angeles Unified']

,0,1,2,3,4,5,6
293,19,64733,Los Angeles Unified,10877387458.05,371342.1,29292.09,Unified


## 2. SACS statement: kind of dollar × function (General Fund operating spending)

In [3]:
gl = sacs_parser.general_ledger(cds, '2425')
sacs_statement.spending_matrix(gl).map(fmt)

function,1000 Instruction,2000 Instruction-Related Services,3000 Pupil Services,4000 Ancillary Services,5000 Community Services,7000 General Administration,8000 Plant Services,9000 Other Outgo,Total
kind,,,,,,,,,
Federal,$237.8M,$337.2M,$110.9M,$14.0M,$0.0M,$37.2M,$0.2M,$0.0M,$737.3M
Local restricted,$32.0M,$7.6M,$45.4M,$0.9M,$1.9M,$21.9M,$354.6M,$0.0M,$464.3M
State restricted (2000s),$321.6M,$88.0M,$10.2M,$99.6M,$0.0M,$32.2M,$86.9M,$0.0M,$638.6M
State restricted (6000s-7000s),"$1,770.6M",$300.4M,$790.8M,$106.8M,$0.1M,$127.7M,$1.0M,$0.0M,"$3,097.5M"
Unrestricted (LCFF),"$3,294.7M","$1,062.6M",$425.7M,$39.8M,$4.0M,$505.6M,$687.4M,$0.9M,"$6,020.6M"
Total,"$5,656.7M","$1,795.8M","$1,383.0M",$261.1M,$6.0M,$724.6M,"$1,130.1M",$0.9M,"$10,958.2M"


In [4]:
sacs_statement.top_resources(gl).assign(amount=lambda d: d.amount.map(fmt))

,Resource,title,kind,amount
0,0000,Unrestricted,Unrestricted (LCFF),"$4,968.7M"
1,6500,Special Education,State restricted (6000s-7000s),"$1,625.0M"
2,1400,Education Protection Account,Unrestricted (LCFF),$975.2M
3,2600,Expanded Learning Opportunities Program,State restricted (2000s),$638.6M
4,7435,Learning Recovery Emergency Block Grant,State restricted (6000s-7000s),$637.3M
5,3010,"ESSA: Title I, Part A, Basic Grants Low-Income...",Federal,$434.9M
6,8150,Ongoing & Major Maintenance Account (RMA: Educ...,Local restricted,$370.3M
7,7690,On-Behalf Pension Contributions,State restricted (6000s-7000s),$368.9M
8,3310,Special Ed: IDEA Basic Local Assistance Entitl...,Federal,$116.4M
9,6010,After School Education and Safety (ASES),State restricted (6000s-7000s),$100.3M


## 3. School model: pick the outcome
`OUTCOME` is the same switch the dashboard will use: `'level'` (2025 % met) or `'growth'` (2025 minus 2024, controlling for the 2024 score). `python src/returns_model.py --outcome both` writes both into one table with an `outcome` column.

In [5]:
OUTCOME = 'growth'   # or 'level'
schools, coefs, stats, excluded = returns_model.run(cds, OUTCOME)
stats.round(3)

,outcome,subject,n,r2_controls_only,r2_with_spending,residual_sd_pts
0,growth,ela,649,0.189,0.199,5.149
1,growth,math,649,0.132,0.135,5.507


### Method A: district par ROI, points per $1,000/pupil by kind of dollar

In [6]:
coefs.round(3)

,outcome,subject,kind_of_dollar,par_roi_per_1000,ci_low,ci_high,p_value
0,growth,ela,Federal,0.430,-0.263,1.123,0.224
1,growth,ela,State & local,-0.191,-0.337,-0.045,0.011
2,growth,ela,All dollars,-0.120,-0.243,0.003,0.056
3,growth,math,Federal,0.463,-0.400,1.326,0.293
4,growth,math,State & local,-0.080,-0.249,0.089,0.352
5,growth,math,All dollars,-0.019,-0.142,0.104,0.764


### Method B: each school vs par (per $10,000/pupil), and Method C peer cross-check

In [7]:
cols = ['school_name','span','total_ppe','actual','par','vs_par','roi_per_10k','par_roi_per_10k','band','peer_par_roi_per_10k','peer_band']
ela = schools[schools.subject == 'ela']
print(f"B vs C agreement: {(ela.band == ela.peer_band).mean():.1%}")
pd.concat([ela.nsmallest(3, 'z'), ela.nlargest(3, 'z')])[cols].round(2)

B vs C agreement: 76.3%


,school_name,span,total_ppe,actual,par,vs_par,roi_per_10k,par_roi_per_10k,band,peer_par_roi_per_10k,peer_band
589,Valley View Elementary,elementary,26859,-19.71,-2.38,-17.33,-7.34,-0.89,below par,0.87,below par
35,Avalon Gardens Elementary,elementary,37119,-12.13,4.46,-16.59,-3.27,1.20,below par,0.77,below par
625,Westport Heights Elementary,elementary,31221,-10.97,4.60,-15.57,-3.51,1.47,below par,1.30,below par
306,Jordan High,high,27892,28.88,9.75,19.13,10.35,3.50,above par,1.88,above par
147,Diego Rivera Learning Complex Green Design STE...,high,21199,27.09,10.67,16.42,12.78,5.03,above par,1.88,above par
273,Hollywood Senior High,high,27633,16.56,1.45,15.11,5.99,0.53,above par,0.62,above par


In [8]:
schools.groupby('subject')['band'].value_counts().unstack()

band,above par,below par,par
subject,,,
ela,181,189,279
math,178,194,277


In [9]:
excluded['reason'].value_counts()

reason
non-traditional school type                      114
fewer than 30 tested / no scores                  18
no 2024 score or fewer than 30 tested in 2024      2
Name: count, dtype: int64